# Module 12 — Alpha Validation

Diagnostic-only evaluation of whether the original static OOS profitability provides evidence of alpha. This module does not modify or tune the strategy.

It tests three distinct claims: (1) positive market-adjusted alpha using HAC inference, (2) robustness of the realized daily mean return using a moving-block bootstrap, and (3) whether the structural Top40 selection outperforms random 40-pair selections from the same formation-sample H < 0.5 pool.

The pair-selection placebo is the main strategy-specific test. Failure to reject a null does not prove luck; it means the OOS sample does not provide sufficiently strong evidence against that null.

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf
from src.backtest import run_walk_forward_backtest, backtest_summary
from src.pair_eligibility import filter_antipersistent_pairs

pd.set_option('display.max_columns', 120)
OUT = Path('data/processed/alpha_validation')
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
INITIAL_CAPITAL = 100000.0


## 1. Load original static inputs and outputs

In [ ]:
train_prices = pd.read_parquet('data/processed/train_prices.parquet')
test_prices = pd.read_parquet('data/processed/test_prices.parquet')
top40 = pd.read_parquet('data/processed/eligible_pairs.parquet')
cointegrated_pairs = pd.read_parquet('data/processed/cointegrated_pairs.parquet')
fou_parameters = pd.read_parquet('data/processed/fractional_ou_parameters.parquet')
rf_data = pd.read_parquet('data/processed/risk_free_rates.parquet')
risk_free_rates = rf_data.iloc[:, 0] if isinstance(rf_data, pd.DataFrame) else pd.Series(rf_data)
risk_free_rates.index = pd.to_datetime(risk_free_rates.index)
risk_free_rates = risk_free_rates.sort_index().astype(float)
equity = pd.read_parquet('data/processed/equity_curve.parquet').copy()
if 'date' in equity.columns:
    equity['date'] = pd.to_datetime(equity['date'])
    equity = equity.set_index('date')
equity.index = pd.to_datetime(equity.index).tz_localize(None)
equity = equity.sort_index()
print('Original selected pairs:', len(top40))
print('OOS equity observations:', len(equity))


## 2. Market-adjusted alpha with HAC/Newey-West inference

Estimate R_strategy - R_f = alpha + beta_M (R_market - R_f) + error. The one-sided p-value tests H0: alpha <= 0 against H1: alpha > 0.

In [ ]:
start = equity.index.min()
end = equity.index.max() + pd.Timedelta(days=1)
spy = yf.download('SPY', start=start.strftime('%Y-%m-%d'), end=end.strftime('%Y-%m-%d'), auto_adjust=True, progress=False)
spy_close = spy['Close'].iloc[:, 0] if isinstance(spy.columns, pd.MultiIndex) else spy['Close']
spy_close.index = pd.to_datetime(spy_close.index).tz_localize(None)
strategy_ret = equity['equity'].pct_change().rename('strategy_return')
market_ret = spy_close.pct_change().rename('market_return')
rf_daily = (risk_free_rates.reindex(strategy_ret.index, method='ffill') / 252.0).rename('rf_daily')
reg = pd.concat([strategy_ret, market_ret, rf_daily], axis=1).dropna()
reg['strategy_excess'] = reg['strategy_return'] - reg['rf_daily']
reg['market_excess'] = reg['market_return'] - reg['rf_daily']
fit = sm.OLS(reg['strategy_excess'], sm.add_constant(reg['market_excess'])).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
alpha_daily = float(fit.params['const'])
alpha_t = float(fit.tvalues['const'])
alpha_p_two = float(fit.pvalues['const'])
alpha_p_one = alpha_p_two / 2 if alpha_t > 0 else 1 - alpha_p_two / 2
factor_alpha = pd.DataFrame([{'alpha_daily': alpha_daily, 'alpha_annualized_simple': alpha_daily * 252, 'alpha_HAC_se': float(fit.bse['const']), 'alpha_t_stat': alpha_t, 'alpha_p_value_two_sided': alpha_p_two, 'alpha_p_value_one_sided_positive': alpha_p_one, 'market_beta': float(fit.params['market_excess']), 'r_squared': float(fit.rsquared), 'n_obs': int(fit.nobs)}])
display(factor_alpha)
factor_alpha.to_csv(OUT / 'market_adjusted_alpha.csv', index=False)


## 3. Moving-block bootstrap of the daily mean return

Resample 20-trading-day blocks rather than individual days so serial dependence is not discarded. The implementation below vectorizes the bootstrap in batches to avoid a slow Python-level loop.

In [ ]:
returns = strategy_ret.dropna().to_numpy(float)
BLOCK_LENGTH = 20
N_BOOTSTRAP = 10000
BOOTSTRAP_BATCH = 1000
rng = np.random.default_rng(SEED)
n = len(returns)
n_blocks = int(np.ceil(n / BLOCK_LENGTH))
offsets = np.arange(BLOCK_LENGTH)
bootstrap_means = np.empty(N_BOOTSTRAP, dtype=float)

for lo in range(0, N_BOOTSTRAP, BOOTSTRAP_BATCH):
    hi = min(lo + BOOTSTRAP_BATCH, N_BOOTSTRAP)
    starts = rng.integers(0, n - BLOCK_LENGTH + 1, size=(hi - lo, n_blocks))
    idx = starts[..., None] + offsets
    samples = returns[idx].reshape(hi - lo, -1)[:, :n]
    bootstrap_means[lo:hi] = samples.mean(axis=1)

ci_low, ci_high = np.quantile(bootstrap_means, [0.025, 0.975])
observed_mean = float(returns.mean())
bootstrap_summary = pd.DataFrame([{'observed_mean_daily_return': observed_mean, 'observed_annualized_simple_mean': observed_mean * 252, 'bootstrap_ci_2_5_daily': float(ci_low), 'bootstrap_ci_97_5_daily': float(ci_high), 'bootstrap_ci_2_5_annualized_simple': float(ci_low * 252), 'bootstrap_ci_97_5_annualized_simple': float(ci_high * 252), 'block_length_trading_days': BLOCK_LENGTH, 'n_bootstrap': N_BOOTSTRAP}])
display(bootstrap_summary)
bootstrap_summary.to_csv(OUT / 'moving_block_bootstrap.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(bootstrap_means, bins=40, alpha=0.75)
ax.axvline(0.0, linestyle='--', linewidth=1.5, label='Zero')
ax.axvline(observed_mean, linestyle='-', linewidth=2, label='Observed mean')
ax.set_xlabel('Mean daily return')
ax.set_ylabel('Bootstrap frequency')
ax.set_title('Moving-block bootstrap of strategy mean return')
ax.legend()
plt.show()


## 4. Structural Top40 pair-selection placebo

Each placebo samples 40 pairs without replacement from the same formation-sample H < 0.5 pool and runs the identical OOS trading machinery. To reduce runtime while preserving the 100-portfolio randomization sample, placebo backtests are evaluated concurrently and use 500 conditional fOU paths per signal. The actual Top40 benchmark is rerun at the same 500-path precision, so the randomization comparison remains apples-to-apples. The canonical strategy backtest remains unchanged at its original simulation precision.

In [ ]:
H_POOL = filter_antipersistent_pairs(fou_parameters).drop_duplicates('pair').reset_index(drop=True)
N_PAIRS = len(top40)
N_PLACEBOS = 100
PLACEBO_SIGNAL_PATHS = 500
MAX_WORKERS = min(4, os.cpu_count() or 1)

print('H < 0.5 formation pool:', len(H_POOL))
print('Pairs per portfolio:', N_PAIRS)
print('Placebo portfolios:', N_PLACEBOS)
print('Conditional paths per signal:', PLACEBO_SIGNAL_PATHS)
print('Parallel workers:', MAX_WORKERS)

def run_portfolio(pair_frame):
    res = run_walk_forward_backtest(
        train_prices=train_prices,
        test_prices=test_prices,
        eligible_pairs=pair_frame,
        cointegrated_pairs=cointegrated_pairs,
        risk_free_rates=risk_free_rates,
        initial_capital=INITIAL_CAPITAL,
        entry_z=1.5,
        target_probability=0.70,
        memory_window=60,
        max_horizon_days=126,
        n_paths=PLACEBO_SIGNAL_PATHS,
        ewma_lambda=0.94,
        seed=SEED,
    )
    eq = res['equity_curve'].copy()
    tr = res['trades'].copy()
    s = backtest_summary(tr, eq, INITIAL_CAPITAL)
    daily = eq['equity'].pct_change().dropna()
    daily_std = daily.std(ddof=1)
    return {
        'total_return': float(s['total_return']),
        'max_drawdown': float(s['max_drawdown']),
        'final_equity': float(s['final_equity']),
        'n_trades': int(len(tr)),
        'mean_trade_return': float(tr['trade_return'].mean()) if len(tr) else np.nan,
        'total_trade_pnl': float(tr['pnl'].sum()) if len(tr) else 0.0,
        'daily_sharpe': float(np.sqrt(252) * daily.mean() / daily_std) if daily_std > 0 else np.nan,
    }

actual_same_mc = run_portfolio(top40)
print('Actual Top40 benchmark at placebo MC precision:')
display(pd.Series(actual_same_mc).to_frame('value'))


In [ ]:
rng = np.random.default_rng(SEED)
samples = []
for b in range(N_PLACEBOS):
    idx = rng.choice(len(H_POOL), size=N_PAIRS, replace=False)
    samples.append((b, H_POOL.iloc[idx].copy()))

def run_one_placebo(item):
    b, sampled = item
    stats = run_portfolio(sampled)
    stats['placebo_id'] = int(b)
    stats['sampled_pairs'] = '|'.join(sorted(sampled['pair'].astype(str)))
    return stats

placebo_rows = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for i, stats in enumerate(executor.map(run_one_placebo, samples), start=1):
        placebo_rows.append(stats)
        if i % 10 == 0 or i == N_PLACEBOS:
            print(f'Completed {i}/{N_PLACEBOS}')

placebos = pd.DataFrame(placebo_rows).sort_values('placebo_id').reset_index(drop=True)
placebos.to_parquet(OUT / 'pair_selection_placebos.parquet', index=False)
display(placebos.describe(include='all'))


## 5. Randomization test

For metrics where larger is better, p = (1 + number of placebo metrics >= actual metric) / (B + 1). The primary metric is total OOS return; Sharpe and trade P&L are supporting diagnostics.

In [ ]:
rows = []
for metric in ['total_return', 'daily_sharpe', 'total_trade_pnl', 'mean_trade_return']:
    actual = float(actual_same_mc[metric])
    null = placebos[metric].dropna().to_numpy(float)
    p = (1 + np.sum(null >= actual)) / (len(null) + 1)
    rows.append({'metric': metric, 'actual': actual, 'placebo_mean': float(null.mean()), 'placebo_median': float(np.median(null)), 'one_sided_randomization_p_value': float(p), 'actual_percentile_vs_placebos': float(100 * np.mean(null < actual))})

placebo_test = pd.DataFrame(rows)
display(placebo_test)
placebo_test.to_csv(OUT / 'pair_selection_randomization_test.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(placebos['total_return'], bins=25, alpha=0.75)
ax.axvline(actual_same_mc['total_return'], linestyle='--', linewidth=2, label='Actual Top40')
ax.set_xlabel('OOS total return')
ax.set_ylabel('Placebo frequency')
ax.set_title('Actual structural Top40 vs random H < 0.5 selections')
ax.legend()
plt.show()


## 6. Alpha-validation scorecard

In [ ]:
alpha_pass = bool(factor_alpha.loc[0, 'alpha_p_value_one_sided_positive'] < 0.05)
bootstrap_pass = bool(bootstrap_summary.loc[0, 'bootstrap_ci_2_5_daily'] > 0)
ret_row = placebo_test.loc[placebo_test['metric'].eq('total_return')].iloc[0]
placebo_pass = bool(ret_row['one_sided_randomization_p_value'] < 0.05)

scorecard = pd.DataFrame([
    {'test': 'HAC positive alpha', 'reject_null_at_5pct': alpha_pass, 'evidence': f"alpha={factor_alpha.loc[0, 'alpha_daily']:.6f}/day, one-sided p={factor_alpha.loc[0, 'alpha_p_value_one_sided_positive']:.4f}"},
    {'test': '20-day moving-block bootstrap', 'reject_null_at_5pct': bootstrap_pass, 'evidence': f"95% CI daily mean=[{bootstrap_summary.loc[0, 'bootstrap_ci_2_5_daily']:.6f}, {bootstrap_summary.loc[0, 'bootstrap_ci_97_5_daily']:.6f}]"},
    {'test': 'Structural Top40 pair-selection placebo', 'reject_null_at_5pct': placebo_pass, 'evidence': f"actual return={ret_row['actual']:.3%}, randomization p={ret_row['one_sided_randomization_p_value']:.4f}"},
])
display(scorecard)
scorecard.to_csv(OUT / 'alpha_validation_scorecard.csv', index=False)
